# Session 2, Module 07: Inheritance and Polymorphism


This module covers:
- Single inheritance
- Method overriding
- super() and MRO (Method Resolution Order)
- Abstract base classes (abc module, @abstractmethod)
- Polymorphism in practice

Data Engineering Context:
Inheritance lets you build a family of extractors (API, Database, File)
that share a common interface but have specialized implementations.


In [1]:
from abc import ABC, abstractmethod
from typing import Any

## Single Inheritance


In [2]:
print("=== Single Inheritance ===")


class DataSource:
    """Base class for all data sources."""

    def __init__(self, name: str):
        self.name = name
        self.is_connected = False

    def connect(self) -> bool:
        """Establish connection to data source."""
        print(f"Connecting to {self.name}...")
        self.is_connected = True
        return True

    def disconnect(self) -> None:
        """Close connection."""
        print(f"Disconnecting from {self.name}")
        self.is_connected = False

    def get_info(self) -> str:
        """Return information about the data source."""
        return f"DataSource: {self.name}"


class DatabaseSource(DataSource):
    """Database-specific data source. Inherits from DataSource."""

    def __init__(self, name: str, host: str, port: int, database: str):
        # Call parent __init__
        super().__init__(name)

        # Add child-specific attributes
        self.host = host
        self.port = port
        self.database = database

    def get_info(self) -> str:
        """Override to include database details."""
        return f"DatabaseSource: {self.name} ({self.host}:{self.port}/{self.database})"

    def execute_query(self, query: str) -> list:
        """Database-specific method."""
        if not self.is_connected:
            raise RuntimeError("Not connected")
        print(f"Executing query: {query[:50]}...")
        return [{"id": 1, "data": "sample"}]


# Create instances
base_source = DataSource("generic")
db_source = DatabaseSource("warehouse", "localhost", 5432, "analytics")

# Both have connect() from parent
base_source.connect()
db_source.connect()

# get_info() is overridden in child
print(f"\n{base_source.get_info()}")
print(f"{db_source.get_info()}")

# Child has additional methods
result = db_source.execute_query("SELECT * FROM customers")
print(f"Query result: {result}")

# Check inheritance
print(f"\nIsinstance checks:")
print(f"  db_source is DataSource: {isinstance(db_source, DataSource)}")
print(f"  db_source is DatabaseSource: {isinstance(db_source, DatabaseSource)}")
print(f"  base_source is DatabaseSource: {isinstance(base_source, DatabaseSource)}")

=== Single Inheritance ===
Connecting to generic...
Connecting to warehouse...

DataSource: generic
DatabaseSource: warehouse (localhost:5432/analytics)
Executing query: SELECT * FROM customers...
Query result: [{'id': 1, 'data': 'sample'}]

Isinstance checks:
  db_source is DataSource: True
  db_source is DatabaseSource: True
  base_source is DatabaseSource: False


============================================================
super() AND METHOD RESOLUTION ORDER (MRO)
============================================================

In [3]:
print("\n=== super() and MRO ===")


class Base:
    def __init__(self):
        print("Base.__init__()")
        self.base_attr = "from base"

    def method(self):
        print("Base.method()")


class Child(Base):
    def __init__(self):
        print("Child.__init__() - before super()")
        super().__init__()  # Call parent's __init__
        print("Child.__init__() - after super()")
        self.child_attr = "from child"

    def method(self):
        print("Child.method() - before super()")
        super().method()  # Call parent's method
        print("Child.method() - after super()")


print("Creating Child instance:")
child = Child()
print(f"\nCalling method:")
child.method()

# MRO - Method Resolution Order
print(f"\nMRO for Child: {Child.__mro__}")


=== super() and MRO ===
Creating Child instance:
Child.__init__() - before super()
Base.__init__()
Child.__init__() - after super()

Calling method:
Child.method() - before super()
Base.method()
Child.method() - after super()

MRO for Child: (<class '__main__.Child'>, <class '__main__.Base'>, <class 'object'>)


Shows: (Child, Base, object) - search order for methods
Multiple inheritance MRO

In [4]:
class A:
    def method(self):
        print("A.method")


class B(A):
    def method(self):
        print("B.method")
        super().method()


class C(A):
    def method(self):
        print("C.method")
        super().method()


class D(B, C):  # Multiple inheritance
    def method(self):
        print("D.method")
        super().method()


print(f"\nMRO for D: {[cls.__name__ for cls in D.__mro__]}")
# D -> B -> C -> A -> object (linearized)

print("\nCalling D().method():")
D().method()


MRO for D: ['D', 'B', 'C', 'A', 'object']

Calling D().method():
D.method
B.method
C.method
A.method


## Abstract Base Classes (Abc)


In [5]:
print("\n=== Abstract Base Classes ===")


class BaseExtractor(ABC):
    """
    Abstract base class for data extractors.

    Subclasses MUST implement extract() method.
    This enforces a contract/interface.
    """

    def __init__(self, name: str):
        self.name = name
        self._records_extracted = 0

    @abstractmethod
    def extract(self) -> list[dict]:
        """
        Extract data from source.

        Returns:
            List of records as dictionaries.
        """
        pass  # Subclasses must implement

    @abstractmethod
    def validate_connection(self) -> bool:
        """Validate that connection to source is valid."""
        pass

    # Concrete method - implemented in base, inherited by all
    def get_stats(self) -> dict:
        """Return extraction statistics."""
        return {
            "name": self.name,
            "records_extracted": self._records_extracted,
        }


=== Abstract Base Classes ===


Cannot instantiate abstract class
extractor = BaseExtractor("test")  # TypeError!

In [6]:
class APIExtractor(BaseExtractor):
    """Extract data from REST API."""

    def __init__(self, name: str, base_url: str, endpoint: str):
        super().__init__(name)
        self.base_url = base_url
        self.endpoint = endpoint

    def validate_connection(self) -> bool:
        """Check API is accessible."""
        print(f"Validating API: {self.base_url}")
        return True

    def extract(self) -> list[dict]:
        """Fetch data from API endpoint."""
        print(f"Extracting from {self.base_url}/{self.endpoint}")
        # Simulate API response
        data = [{"id": 1, "name": "API Record"}]
        self._records_extracted = len(data)
        return data


class DatabaseExtractor(BaseExtractor):
    """Extract data from database."""

    def __init__(self, name: str, connection_string: str, query: str):
        super().__init__(name)
        self.connection_string = connection_string
        self.query = query

    def validate_connection(self) -> bool:
        """Check database connection."""
        print(f"Validating DB: {self.connection_string}")
        return True

    def extract(self) -> list[dict]:
        """Execute query and return results."""
        print(f"Executing: {self.query[:30]}...")
        # Simulate query results
        data = [{"id": 1, "name": "DB Record"}, {"id": 2, "name": "DB Record 2"}]
        self._records_extracted = len(data)
        return data


class FileExtractor(BaseExtractor):
    """Extract data from files."""

    def __init__(self, name: str, file_path: str, file_format: str = "csv"):
        super().__init__(name)
        self.file_path = file_path
        self.file_format = file_format

    def validate_connection(self) -> bool:
        """Check file exists and is readable."""
        print(f"Validating file: {self.file_path}")
        return True

    def extract(self) -> list[dict]:
        """Read data from file."""
        print(f"Reading {self.file_format} from {self.file_path}")
        # Simulate file read
        data = [{"id": 1, "name": "File Record"}]
        self._records_extracted = len(data)
        return data


# Create different extractors
api_extractor = APIExtractor("api_users", "https://api.example.com", "users")
db_extractor = DatabaseExtractor("db_orders", "postgresql://...", "SELECT * FROM orders")
file_extractor = FileExtractor("file_products", "/data/products.csv")

extractors = [api_extractor, db_extractor, file_extractor]

## Polymorphism — Same Interface, Different Behavior


In [7]:
print("\n=== Polymorphism ===")


def run_extraction(extractor: BaseExtractor) -> list[dict]:
    """
    Run extraction on any extractor type.

    This function works with ANY subclass of BaseExtractor
    because they all implement the same interface.
    """
    print(f"\n--- Extracting: {extractor.name} ---")

    # Validate (each subclass has its own implementation)
    if not extractor.validate_connection():
        raise RuntimeError(f"Validation failed for {extractor.name}")

    # Extract (each subclass has its own implementation)
    data = extractor.extract()

    # Get stats (inherited from base class)
    stats = extractor.get_stats()
    print(f"Stats: {stats}")

    return data


# Same function works with different extractor types!
for extractor in extractors:
    data = run_extraction(extractor)
    print(f"Extracted {len(data)} records")


=== Polymorphism ===

--- Extracting: api_users ---
Validating API: https://api.example.com
Extracting from https://api.example.com/users
Stats: {'name': 'api_users', 'records_extracted': 1}
Extracted 1 records

--- Extracting: db_orders ---
Validating DB: postgresql://...
Executing: SELECT * FROM orders...
Stats: {'name': 'db_orders', 'records_extracted': 2}
Extracted 2 records

--- Extracting: file_products ---
Validating file: /data/products.csv
Reading csv from /data/products.csv
Stats: {'name': 'file_products', 'records_extracted': 1}
Extracted 1 records


## Practical: Extractor Factory


In [8]:
print("\n=== Practical: Extractor Factory ===")


class ExtractorFactory:
    """Factory to create appropriate extractor based on config."""

    _extractors: dict[str, type] = {
        "api": APIExtractor,
        "database": DatabaseExtractor,
        "file": FileExtractor,
    }

    @classmethod
    def register(cls, name: str, extractor_class: type) -> None:
        """Register a new extractor type."""
        cls._extractors[name] = extractor_class

    @classmethod
    def create(cls, config: dict) -> BaseExtractor:
        """
        Create extractor from configuration.

        Args:
            config: Dict with 'type' and type-specific parameters
        """
        extractor_type = config.get("type")
        if extractor_type not in cls._extractors:
            raise ValueError(f"Unknown extractor type: {extractor_type}")

        extractor_class = cls._extractors[extractor_type]

        # Remove 'type' and pass rest as kwargs
        params = {k: v for k, v in config.items() if k != "type"}
        return extractor_class(**params)


# Configuration-driven extractor creation
configs = [
    {
        "type": "api",
        "name": "users_api",
        "base_url": "https://api.example.com",
        "endpoint": "users",
    },
    {
        "type": "database",
        "name": "orders_db",
        "connection_string": "postgresql://localhost/db",
        "query": "SELECT * FROM orders",
    },
    {
        "type": "file",
        "name": "products_file",
        "file_path": "/data/products.csv",
    },
]

print("Creating extractors from config:")
for config in configs:
    extractor = ExtractorFactory.create(config)
    print(f"  Created: {extractor.name} ({type(extractor).__name__})")


=== Practical: Extractor Factory ===
Creating extractors from config:
  Created: users_api (APIExtractor)
  Created: orders_db (DatabaseExtractor)
  Created: products_file (FileExtractor)


## Summary


In [10]:
print("\n=== Summary ===")
print("""
Inheritance:
  class Child(Parent):
      def __init__(self):
          super().__init__()  # Call parent

Method Overriding:
  - Child can redefine parent methods
  - Use super() to call parent version
  - MRO determines search order

Abstract Base Classes:
  from abc import ABC, abstractmethod

  class Base(ABC):
      @abstractmethod
      def method(self):
          pass  # Must be implemented

  - Cannot instantiate ABC directly
  - Enforces interface contract

Polymorphism:
  - Different classes, same interface
  - Code works with any implementation
  - Enables flexible, extensible design

MRO (Method Resolution Order):
  - ClassName.__mro__ shows search order
  - Python uses C3 linearization
  - super() follows MRO, not just parent

Best Practices:
  - Use ABC for defining interfaces
  - Favor composition over inheritance
  - Keep inheritance hierarchies shallow
  - Use factory pattern for flexibility
""")


=== Summary ===

Inheritance:
  class Child(Parent):
      def __init__(self):
          super().__init__()  # Call parent

Method Overriding:
  - Child can redefine parent methods
  - Use super() to call parent version
  - MRO determines search order

Abstract Base Classes:
  from abc import ABC, abstractmethod

  class Base(ABC):
      @abstractmethod
      def method(self):
          pass  # Must be implemented

  - Cannot instantiate ABC directly
  - Enforces interface contract

Polymorphism:
  - Different classes, same interface
  - Code works with any implementation
  - Enables flexible, extensible design

MRO (Method Resolution Order):
  - ClassName.__mro__ shows search order
  - Python uses C3 linearization
  - super() follows MRO, not just parent

Best Practices:
  - Use ABC for defining interfaces
  - Favor composition over inheritance
  - Keep inheritance hierarchies shallow
  - Use factory pattern for flexibility

